In [ ]:
include( "../geom.jl" )
include( "../kernel.jl" )

_, base = expfolder( "system" );

In [ ]:
# Density and speed parameters.
include( "size-0-params.jl" )

# System size.
Nlist = round.( defInt, 2.0.^(3:7) )
NN = length( Nlist )
println( "Import $(NN) system sizes: $(extrema( Nlist )) (log spacing)." )

# Extract critical noise.
folderdatalist = [[findfolder( N, ρlist[i], slist[j]; base=base, crit=true ) for i ∈ 1:Nρ, j ∈ 1:Ns]
    for N ∈ Nlist];

In [ ]:
# Color map used for plotting.
colormap = cgrad( :ice )

In [ ]:
function decaystrength(x::Vector{defFloat}, y::Vector{defFloat})::Vector{defFloat}
    # x-axis variables.
    X = [ones( length( x ) ) log2.( x )]

    # Return fitted parameters.
    return X \ log2.( y )
end

# Import the coagulation-fragmentation rate coefficients.
γdatalist = [first.( readdlm.( folderdatalist[i].*"kernel-parameters-1.txt" ) ) for i ∈ 1:NN];

# Compute the decay strength for changing speed and density.
αdata = [decaystrength( defFloat.( Nlist ), [γdatalist[k][i,j] for k ∈ 1:NN]./100 ) for i ∈ 1:Nρ, j ∈ 1:Ns]

In [ ]:
# Plot the coagulation-fragmentation ratio across speed and density.
plt = plot(; layout=@layout( [a{0.75w} b] ), size=(250,200) )
plot!( plt; left_margin=-5pt, bottom_margin=-5pt )

heatmap!( plt[1], ρlist, slist, log10.( γdatalist[NN]./100 ); color=colormap, colorbar=false )

plot!( plt[1]; xlims=extrema( ρlist ), xscale=:log10 )
plot!( plt[1]; ylims=extrema( slist ), yscale=:log10 )
plot!( plt[1]; xlabel="density, "*L"ρ", ylabel="speed, "*L"s" )

plot!( plt[2]; xticks=[], yticks=[], framestyle=:box )

In [ ]:
# Plot the coagulation-fragmentation ratio across speed and density.
plt = plot(; layout=@layout( [a{0.75w} b] ), size=(250,200) )
plot!( plt; left_margin=-5pt, bottom_margin=-5pt )

heatmap!( plt[1], ρlist, slist, last.( αdata ); color=colormap, colorbar=false )

plot!( plt[1]; xlims=extrema( ρlist ), xscale=:log10 )
plot!( plt[1]; ylims=extrema( slist ), yscale=:log10 )
plot!( plt[1]; xlabel="density, "*L"ρ", ylabel="speed, "*L"s" )


# Plot the colorbar.
# ymap = 10.0.^(-3:0.0)
# plot!( plt[2] )

plot!( plt[2]; xticks=[], yticks=[], framestyle=:box )

In [ ]:
j = 7
γdata = [[γdatalist[k][i,j] for k ∈ 1:NN]./100 for i ∈ 1:Nρ-1]
println( "Speed: ", slist[j] )

# Initialize plot variable.
plt = plot(; size=(250,275), legend=false )

# Plot the change in the coagulation-fragmentation ratio with system size.
Nlims = vcat( extrema( Nlist )... )
for i ∈ 1:Nρ-1
    # Unpack power-law.
    α = αdata[i,j]
    γ̂ = (2.0^α[1]).*Nlims.^α[2]

    # Plot the data.
    color = colormap1[(i-1)/(Nρ-1)]
    scatter!( plt, Nlist, γdata[i]; color=color )
    plot!( plt, Nlims, γ̂; color=color, lw=2 )
end

xticks = Nlist
xmin = log2( minimum( Nlist ) ) - 1/4
xmax = log2( maximum( Nlist ) ) + 1/4
plot!( plt; xlims=2.0.^(xmin,xmax), xscale=:log10, xticks=(xticks, string.( xticks )) )

ymin = floor( log10( minimum( minimum.( γdata ) ) ) )
ymax = ceil( log10( maximum( maximum.( γdata ) ) ) ) + 1
plot!( plt; ylims=10.0.^(ymin,ymax), yscale=:log10 )

plot!( plt; xlabel="system size, "*L"N", ylabel="coagulation-fragmentation ratio, "*L"γ" )
plot!( plt; framestyle=:box )

# Plot the scaling parameter with changing speed and density.
pltα = plot!( plt; inset=(1, bbox( 0.325, 0.525, 0.40, 0.40, :bottom, :left )) )

heatmap!( plt[2], ρlist, slist, -last.( αdata ); color=colormap2, colorbar=false )

plot!( plt[2]; xlims=extrema( ρlist ), xscale=:log10, xticks=[] )
plot!( plt[2]; ylims=extrema( slist ), yscale=:log10, yticks=[] )
plot!( plt[2]; xlabel=L"ρ \rightarrow", ylabel=L"s \rightarrow" )
plot!( plt[2]; framestyle=:box )

# Plot the scaling parameter with changing speed and density.
pltαc = plot!( plt; inset=(1, bbox( 0.775, 0.525, 0.05, 0.4, :bottom, :left )) )

xmap = [0, 1]
ymap = 0:0.001:1.0
zmap = hcat( ymap, ymap )
heatmap!( plt[3], xmap, ymap, zmap; color=colormap2, colorbar=false )

plot!( plt[3]; xticks=[] )
plot!( plt[3]; ylims=(0,1), yticks=[], ymirror=true )
plot!( plt[3]; ylabel=L"α \rightarrow" )
plot!( plt[3]; framestyle=:box )

plot!( plt; left_margin=0pt, bottom_margin=-5pt, right_margin=-10pt )
plot!( plt[1]; title="(a)", titleposition=:left )

# saveplot( plt, figurefolder*"cf-ratio-scaling_N-$(Nlist[1])-$(Nlist[end])"; background=:white, dpi=600 );

In [ ]:
i = 7
γdata = [[γdatalist[k][i,j] for k ∈ 1:NN]./100 for j ∈ 1:Ns-1]
println( "Density: ", ρlist[i] )

# Plot the change in the coagulation-fragmentation ratio with system size.
plt = plot(; size=(300,250), legend=false )

Nlims = vcat( extrema( Nlist )... )
for j ∈ 1:Ns-1
    # Unpack power-law.
    α = αdata[i,j]
    γ̂ = (2.0^α[1]).*Nlims.^α[2]

    # Plot the data.
    color = colormap2[(j-1)/(Ns-1)]
    scatter!( plt, Nlist, γdata[j]; color=color )
    plot!( plt, Nlims, γ̂; color=color, lw=2 )
end

xticks = Nlist
xmin = log2( minimum( Nlist ) ) - 1/4
xmax = log2( maximum( Nlist ) ) + 1/4
ymin = floor( log10( minimum( minimum.( γdata ) ) ) )
ymax = ceil( log10( maximum( maximum.( γdata ) ) ) ) + 1
plot!( plt; ylims=10.0.^(ymin,ymax), yscale=:log10 )
plot!( plt; xlims=2.0.^(xmin,xmax), xscale=:log10, xticks=(xticks, string.( xticks )) )
plot!( plt; xlabel="system size, "*L"N", ylabel="coagulation-\nfragmentation ratio, "*L"γ" )
plot!( plt; framestyle=:box )

In [ ]:
if false
    # Save the decay strength (indexed by the maximum size considered).
    for i ∈ 1:Nρ, j ∈ 1:Ns
        writedlm( folderdatalist[NN][i,j]*"kernel-decay-strength.txt", αdata[i,j] )
    end
end